In [1]:
# =====================================================================
# MSAC-Xray: Complete End-to-End Google Colab Pipeline
# =====================================================================

import os
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import DenseNet121, ResNet50, MobileNetV2
from tensorflow.keras.applications.densenet import preprocess_input as pre_dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# 1. Random Seed Stabilization for Full Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("🔒 System random seed fixed successfully (Seed: 42).")

# 2. Synthetic Radiographic Data Generation
print("🔄 Generating synthetic X-ray dataset...")
X_list, y_list = [], []

for i in range(200):
    is_fractured = i >= 100
    img = np.random.normal(120, 25, (224, 224)).astype(np.float32)
    img = cv2.GaussianBlur(img, (15, 15), 5)
    cv2.ellipse(img, (112, 112), (75, 45), 30, 0, 360, (60,), -1)

    if is_fractured:
        cv2.line(img, (90, 75), (130, 150), (15,), thickness=2)

    img = np.clip(img + np.random.normal(0, 8, img.shape), 0, 255)
    X_list.append(cv2.resize(img.astype(np.uint8), (224, 224)))
    y_list.append(1 if is_fractured else 0)

X_data = np.array(X_list, dtype=np.uint8)
y_data = np.array(y_list, dtype=np.int32)

X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=SEED, stratify=y_data
)
print(f"📊 Training samples: {len(X_train)} | Test samples: {len(X_test)}")

# 3. MSAC-Xray Preprocessing Pipeline
def msac_xray(img, sigma_s=None):
    I = img.astype(np.float32)
    H, W = I.shape[:2]
    if sigma_s is None:
        sigma_s = 0.15 * np.hypot(H, W)
    k = int(2 * np.ceil(3 * sigma_s) + 1)

    B = cv2.GaussianBlur(I, (k, k), sigma_s)
    mu_B = B.mean()
    I1 = np.clip(I / (B + 1e-6) * mu_B, 0, 255)

    lap = cv2.Laplacian(I1.astype(np.float32), cv2.CV_32F)
    sigma_hat = 1.4826 * np.median(np.abs(lap - np.median(lap)))
    c_max, c_min = 4.0, 1.0
    alpha = (c_max - c_min) / 30.0
    clip = float(np.clip(c_max - alpha * sigma_hat, c_min, c_max))
    t = 16 if sigma_hat > 12.0 else (4 if sigma_hat < 4.0 else 8)

    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(t, t))
    I3 = clahe.apply(np.clip(I1, 0, 255).astype(np.uint8)).astype(np.float32)

    J0 = cv2.bilateralFilter(I3.astype(np.uint8), d=7, sigmaColor=40, sigmaSpace=40)
    p1, p99 = np.percentile(J0, [1, 99])
    J = np.clip((J0.astype(np.float32) - p1) / (p99 - p1 + 1e-6), 0, 1)
    return (J * 255).astype(np.uint8)

# 4. Model Builder with Fine-Tuning
def build_model(name, input_shape=(224, 224, 3), n_classes=1):
    weights = "imagenet"
    if name == "densenet121":
        base = DenseNet121(include_top=False, weights=weights, input_shape=input_shape)
        pre = pre_dense
    elif name == "resnet50":
        base = ResNet50(include_top=False, weights=weights, input_shape=input_shape)
        from tensorflow.keras.applications.resnet50 import preprocess_input as pre
    elif name == "mobilenetv2":
        base = MobileNetV2(include_top=False, weights=weights, input_shape=input_shape)
        from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as pre

    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(n_classes, activation="sigmoid")(x)
    model = Model(base.input, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model, pre

# 5. Training and Evaluation Loop
results_summary = {}

def train_eval(name, X_tr, y_tr, X_te, y_te, epochs=5, batch=16):
    print(f"\n🚀 Training model: {name.upper()} ...")
    model, pre = build_model(name)

    X_tr_p = pre(np.stack([cv2.cvtColor(msac_xray(x), cv2.COLOR_GRAY2RGB) for x in X_tr]))
    X_te_p = pre(np.stack([cv2.cvtColor(msac_xray(x), cv2.COLOR_GRAY2RGB) for x in X_te]))

    cb = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    model.fit(X_tr_p, y_tr, validation_split=0.15, epochs=epochs,
              batch_size=batch, callbacks=[cb], verbose=2)

    prob = model.predict(X_te_p, verbose=0).ravel()
    y_pred = (prob > 0.5).astype(int)

    acc = accuracy_score(y_te, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_te, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_te, y_pred)

    results_summary[name] = {"Accuracy": acc, "Precision": p, "Recall": r, "F1-Score": f1}
    print(f"📊 Results {name}: Accuracy={acc:.4f} | Precision={p:.3f} | Recall={r:.3f} | F1={f1:.3f}")
    print(f"📉 Confusion Matrix:\n{cm}")

for backbone in ["densenet121", "resnet50", "mobilenetv2"]:
    train_eval(backbone, X_train, y_train, X_test, y_test)

print("\n✨ All models trained and evaluated successfully!")

🔒 System random seed fixed successfully (Seed: 42).
🔄 Generating synthetic X-ray dataset...
📊 Training samples: 160 | Test samples: 40

🚀 Training model: DENSENET121 ...
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
9/9 - 156s - 17s/step - accuracy: 0.4412 - loss: 0.8994 - val_accuracy: 0.4583 - val_loss: 0.8383
Epoch 2/5
9/9 - 1s - 96ms/step - accuracy: 0.5074 - loss: 0.7569 - val_accuracy: 0.4583 - val_loss: 0.7934
Epoch 3/5
9/9 - 1s - 83ms/step - accuracy: 0.5809 - loss: 0.7206 - val_accuracy: 0.4583 - val_loss: 0.7561
Epoch 4/5
9/9 - 1s - 82ms/step - accuracy: 0.5882 - loss: 0.6993 - val_accuracy: 0.4583 - val_loss: 0.7245
Epoch 5/5
9/9 - 1s - 85ms/step - accuracy: 0.6618 - loss: 0.6386 - val_accuracy: 0.5000 - val_loss: 0.6961
📊 Results densenet121: Accuracy=0.4500 | Precision=0.361 | Recall=0.450 | F1=0.345
📉 Confusion Matrix:
[[ 1 19]
 [ 3 17]]

🚀 Training model: RESNET50 ...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
9/9 - 57s - 6s/step - accura

📊 Results mobilenetv2: Accuracy=1.0000 | Precision=1.000 | Recall=1.000 | F1=1.000
📉 Confusion Matrix:
[[20  0]
 [ 0 20]]

✨ All models trained and evaluated successfully!
